# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print basic metadata information
print(f"{metadata.name}: {metadata.description}")

# Print additional key info
print("\nLicense:", metadata.license)
print("Published:", metadata.datePublished)
print("Authors:", metadata.author)
print("Record sets:", metadata.recordSet)
print("Version:", metadata.version)

## 2. Data Overview
Review available record sets, fields, and their IDs.
We use the `@id` for all references to dataset entities. The Croissant schema often exposes record sets via `.recordSet`. For each, inspect available fields and columns.

In [ ]:
# List all record sets in the dataset
record_sets = dataset.record_sets()
print("Record Sets (@id):")
for rs in record_sets:
    print(f"- {rs['@id']} (name: {rs.get('name', '')})")

# For illustration, inspect the first record set's available fields and columns
if record_sets:
    record_set_id = record_sets[0]['@id']
    print(f"\nFields in Record Set {record_set_id}:")
    fields = dataset.fields(record_set=record_set_id)
    for field in fields:
        print(f"  - {field['@id']} (name: {field.get('name', '')})")

    print(f"\nColumns in Record Set {record_set_id}:")
    columns = dataset.columns(record_set=record_set_id)
    for col in columns:
        print(f"  - {col['@id']} (name: {col.get('name', '')}, type: {col.get('dataType', '')})")
else:
    print("No record sets found in the dataset.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

We'll extract all records from each record set using their `@id` (all references are always by `@id`).

In [ ]:
# Build a list of record set @ids
record_set_ids = [rs['@id'] for rs in dataset.record_sets()]
dataframes = {}

for rs_id in record_set_ids:
    print(f"Extracting records for record set {rs_id}...")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"  Columns: {df.columns.tolist()}")
    print(f"  Top 5 Records:\n{df.head()}\n")

# For further analysis, choose one record set
if record_set_ids:
    chosen_record_set_id = record_set_ids[0]
    print(f"Using record set {chosen_record_set_id} for further analysis.")
else:
    chosen_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Operations may include removing outliers, transforming data distributions, or grouping data by key attributes.

We'll demonstrate numeric filtering, normalization, and group-by summarization using columns referenced by their `@id`.

In [ ]:
# If we have a record set, proceed to numeric analysis
if chosen_record_set_id is not None:
    df = dataframes[chosen_record_set_id]
    print(f"Available columns in {chosen_record_set_id}: {df.columns.tolist()}")
    
    # Identify a numeric column based on typical survey fields (example: GAD-7 score, PHQ-9 score)
    possible_numeric_fields = ['gad_7_total_score', 'phq_9_total_score', 'psq_total_score', 'age', 'Age']
    numeric_field_id = None
    for field in possible_numeric_fields:
        if field in df.columns:
            numeric_field_id = field
            break
    
    if numeric_field_id:
        print(f"Selected numeric field: {numeric_field_id}")
        threshold = 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())
        
        # Normalize numeric field
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, norm_col]].head())
        
        # Group by a category, e.g., 'gender', 'marital_status', 'village'
        possible_group_fields = ['gender', 'marital_status', 'village']
        group_field = None
        for gf in possible_group_fields:
            if gf in filtered_df.columns:
                group_field = gf
                break
        
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
            print(f"\nMean {numeric_field_id} grouped by {group_field}:")
            print(grouped_df)
        else:
            print("No suitable group field found.")
    else:
        print("No numeric fields found for analysis.")
else:
    print("No record sets available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Here, we'll plot the distribution of the selected numeric field and its grouping by a categorical field, all referenced by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize numeric distributions if possible
if chosen_record_set_id is not None and numeric_field_id:
    plt.figure(figsize=(7, 5))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If grouping field available, show boxplot
    if group_field:
        plt.figure(figsize=(9, 6))
        sns.boxplot(x=group_field, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.show()
else:
    print("Not enough numeric/group fields to visualize.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Explored metadata and all available record sets using Croissant schema with `mlcroissant`.
- Loaded data from record sets using their `@id` (as required for all entity references).
- Demonstrated filtering, normalization, grouping, and basic visualization for numeric and categorical variables.
- The dataset supports detailed analysis of mental health measures.

For deeper analysis, consult the dataset documentation and Croissant schema for additional fields, entity relationships, and domain-specific insights.